# history

> Read turn logs and durable memory without starting an agent or changing the files.

`History` reads the shared JSONL log from the start on each call. It accepts only newline-terminated object rows. This leaves a trailing row for the writer to finish. The session index supplies only `title` and `roots`. A missing or stale index cannot hide log rows.

`sessions` applies `since` to each session's last timestamp and includes an equal timestamp. Its `limit` keeps the newest sessions. `turns` keeps the last `limit` rows and returns them in chronological order. A zero limit returns no rows. A negative limit raises `ValueError`.

The reader never creates an `Agent`. It never writes files, recovers turns, or rebuilds the session index.

In [ ]:
#| default_exp history

In [ ]:
#| export
import json
from collections import deque
from pathlib import Path

from ramabana import setup
from ramabana.history_core import (_confined, _index_from_log, _log_turn, hydrate_turns, history_rows,
                                   labeled_rows)

In [ ]:
#| export

In [ ]:
#| export
class History:
    "Read turn logs and memory without changing them."

    def __init__(
        self,
        cfg=None,              # config folder; None uses the folder containing `config_path()`
        history_name='agent',  # history file prefix
    ):
        self.cfg = Path(cfg) if cfg is not None else setup.config_path().parent
        self.history_name = history_name

    @property
    def history_path(self):
        "The shared JSONL turn log."
        return self.cfg/f'{self.history_name}-history.jsonl'

    @property
    def sessions_path(self):
        "The optional session metadata index."
        return self.cfg/f'{self.history_name}-sessions.json'

    def _file(self, path):
        "`path` resolved, or `None` when it is not a file inside the config folder: a link out of it is never followed."
        try: found = _confined(self.cfg, str(path.relative_to(self.cfg)))
        except ValueError: return None   # a `history_name` that is an absolute path
        return found if found is not None and found.is_file() else None

    def _metadata(self):
        "The session index's `sessions` dict, or an empty dict when the index is missing, unreadable or malformed."
        if (file := self._file(self.sessions_path)) is None: return {}
        try: data = json.loads(file.read_text())
        except (OSError, ValueError): return {}
        rows = data.get('sessions', {}) if isinstance(data, dict) else {}
        return rows if isinstance(rows, dict) else {}

    def _rows(self):
        "Yield each complete object row of the log, or none when the log is missing or unreadable."
        if (file := self._file(self.history_path)) is None: return
        try: yield from (row for _, _, row in history_rows(file))
        except OSError: return

    def sessions(self, since=None, limit=None):
        "Sessions from complete log rows, newest first. `since` includes sessions at that timestamp."
        if limit is not None and limit < 0: raise ValueError('limit must be non-negative')
        if limit == 0 or (file := self._file(self.history_path)) is None: return []
        try: rows = _index_from_log(file)
        except OSError: return []
        metadata = self._metadata()
        out = []
        for sid, row in rows.items():
            if since is not None and row['last_at'] < since: continue
            sid = str(sid)
            meta = metadata.get(sid, {})
            if not isinstance(meta, dict): meta = {}
            roots, title = meta.get('roots'), meta.get('title')
            out.append({'id': sid,
                        'title': title if isinstance(title, str) and title else row.get('first_title', ''),
                        'roots': [r for r in roots if isinstance(r, str)] if isinstance(roots, (list, tuple)) else [],
                        'turns': row['turns'],
                        'first_at': row['first_at'],
                        'at': row['last_at'],
                        'model': row.get('model', '')})
        out.sort(key=lambda row: row['at'], reverse=True)
        return out if limit is None else out[:limit]

    def turns(self, sid, limit=None):
        "One session's hydrated turns in chronological order. `limit` keeps its last rows."
        if limit is not None and limit < 0: raise ValueError('limit must be non-negative')
        if limit == 0: return []
        sid = str(sid)
        out = [] if limit is None else deque(maxlen=limit)
        for row_sid, row in labeled_rows(self._rows()):
            if row_sid == sid: out.append(row)
        return hydrate_turns(self.cfg, list(out))

    def session_meta(self, sid):
        "One session's public metadata, or an empty dict when the log does not contain it."
        return next((row for row in self.sessions() if row['id'] == str(sid)), {})

    def _memory_doc(self, path):
        "`(text, mtime)` of the file at `path`, or `None` when it is not a readable file inside the config folder."
        root = self.cfg.resolve()
        try:
            path = path.resolve()
            if not path.is_relative_to(root) or not path.is_file(): return None
            return path.read_text(errors='replace'), path.stat().st_mtime
        except (OSError, RuntimeError): return None

    def memory(self):
        "Yield each readable durable memory document inside the config folder as `(ref, title, text, mtime)`."
        for p in sorted((self.cfg/'memory').glob('*/MEMORY.md')):
            if (doc := self._memory_doc(p)) is not None:
                yield f'memory/{p.parent.name}', f'ramabana memory ({p.parent.name})', *doc
        learned = self.cfg/'learned'
        agents = learned/'AGENTS.md'
        if (doc := self._memory_doc(agents)) is not None:
            yield 'learned/AGENTS.md', 'ramabana learned: AGENTS.md', *doc
        for p in sorted((learned/'skills').glob('*/SKILL.md')):
            if (doc := self._memory_doc(p)) is not None:
                yield f'learned/skills/{p.parent.name}', f'ramabana learned skill: {p.parent.name}', *doc

`sessions` lists each conversation in a log, newest first. The example writes a log of two conversations into a temporary config folder and reads it back with `History`.

In [ ]:
import tempfile

In [ ]:
cfg = Path(tempfile.mkdtemp())
rows = [{'at': 1, 'session': 'a', 'prompt': 'Fix the parser. Then test it.', 'model': 'gpt-mini'},
        {'at': 2, 'session': 'a', 'prompt': 'Run the tests.', 'model': 'gpt-mini'},
        {'at': 3, 'session': 'b', 'prompt': 'Explain the index', 'model': 'gpt-mini'}]
(cfg/'agent-history.jsonl').write_text(''.join(json.dumps(r) + '\n' for r in rows))
before = sorted(p.name for p in cfg.iterdir())
History(cfg).sessions()

Reading creates no file in the config folder, not even the session index.

In [ ]:
assert sorted(p.name for p in cfg.iterdir()) == before
before